# TF-IDF + Logistic Regression + sublinear_tf=True

Experimento de classificação da clareza das respostas utilizando TF-IDF com ponderação sublinear da frequência dos termos e Regressão Logística.

In [1]:
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

In [2]:
!git clone https://github.com/juliaduboisas/plnEp1.git
%cd plnEp1
!ls

Cloning into 'plnEp1'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 130 (delta 57), reused 86 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 16.18 MiB | 11.97 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/plnEp1
baselineModel.ipynb	     tf_idf_reglog.ipynb
bertimbauModel.ipynb	     tfidfregord.ipynb
dataExploration.ipynb	     tfidf_sublinear.ipynb
exponentialregression.ipynb  TF_IDF_Word_+_Linear_SVM.ipynb
laya_custom		     TF_IDF_+_XGBoost.ipynb
layaFineTune.ipynb	     train.csv
pipelines		     train_noDuplicates.csv
README.md		     train_noSameClassDuplicates.csv
roBERTaModel.ipynb	     useModel.ipynb
sbertModel.ipynb	     word2vecmlp.ipynb
test1_rotulado.xlsx	     word2vecreglog.ipynb
test1.xlsx


## 1. Carregamento dos dados

In [3]:
df = pd.read_csv("train.csv")

Y = df["clarity"]

print(df.shape)
df.head()

(20092, 2)


,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234


## 2. Separação dos dados

In [5]:
x_train_text, x_test_text, y_train, y_test = train_test_split(
    df["resp_text"],
    Y,
    test_size=0.2,
    random_state=123,
    stratify=Y
)

print("Treino:", x_train_text.shape)
print("Teste:", x_test_text.shape)

Treino: (16073,)
Teste: (4019,)


## 3. Vetorização TF-IDF

Neste experimento, utilizamos `sublinear_tf=True`, que aplica uma ponderação sublinear à frequência dos termos.

In [6]:
vect = TfidfVectorizer(
    analyzer="word",
    sublinear_tf=True,
    ngram_range=(1, 3)
)

x_train = vect.fit_transform(x_train_text)
x_test = vect.transform(x_test_text)

print("Treino TF-IDF:", x_train.shape)
print("Teste TF-IDF:", x_test.shape)

Treino TF-IDF: (16073, 1241648)
Teste TF-IDF: (4019, 1241648)


## 4. Regressão Logística

In [10]:
clf = LogisticRegression(
    C=1,
    class_weight="balanced",
    max_iter=1000
)

clf.fit(
    x_train,
    y_train
)

LogisticRegression(C=1, class_weight='balanced', max_iter=1000)

## 5. Avaliação

In [11]:
predicted = clf.predict(x_test)

score = f1_score(
    y_test,
    predicted,
    average="macro"
)

accuracy = accuracy_score(
    y_test,
    predicted
)

print(f"Macro F1: {score:.4f}")
print(f"Accuracy: {accuracy:.4f}")

Macro F1: 0.4774
Accuracy: 0.4775


In [12]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        predicted
    )
)

              precision    recall  f1-score   support

          c1       0.50      0.52      0.51      1269
        c234       0.41      0.39      0.40      1371
          c5       0.52      0.53      0.52      1379

    accuracy                           0.48      4019
   macro avg       0.48      0.48      0.48      4019
weighted avg       0.48      0.48      0.48      4019



In [13]:
# ============================================================
# VALIDAÇÃO CRUZADA — TF-IDF + Logistic Regression
# Melhor configuração encontrada
# ============================================================

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate


# ------------------------------------------------------------
# 1. Dados
# ------------------------------------------------------------

df = pd.read_csv("train.csv")

X = df["resp_text"]
y = df["clarity"]


# ------------------------------------------------------------
# 2. Melhor configuração encontrada
# ------------------------------------------------------------

pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="word",
            sublinear_tf=True,
            ngram_range=(1, 3)
        )
    ),
    (
        "logreg",
        LogisticRegression(
            C=1.0,
            class_weight="balanced"
        )
    )
])


# ------------------------------------------------------------
# 3. Cross-validation estratificada
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)


# ------------------------------------------------------------
# 4. Avaliação
# ------------------------------------------------------------

results = cross_validate(
    pipeline,
    X,
    y,
    cv=cv,
    scoring={
        "f1_macro": "f1_macro",
        "accuracy": "accuracy"
    },
    return_train_score=True,
    n_jobs=-1
)


# ------------------------------------------------------------
# 5. Resultados por fold
# ------------------------------------------------------------

print("Resultados por fold:\n")

for i in range(5):
    print(
        f"Fold {i+1}: "
        f"F1 Macro = {results['test_f1_macro'][i]:.4f} | "
        f"Accuracy = {results['test_accuracy'][i]:.4f}"
    )


# ------------------------------------------------------------
# 6. Média e desvio padrão
# ------------------------------------------------------------

print("\nResultado da validação cruzada:")

print(
    f"Macro F1: "
    f"{results['test_f1_macro'].mean():.4f} "
    f"± {results['test_f1_macro'].std():.4f}"
)

print(
    f"Accuracy: "
    f"{results['test_accuracy'].mean():.4f} "
    f"± {results['test_accuracy'].std():.4f}"
)

print("\nTreino:")
print(
    f"Macro F1: "
    f"{results['train_f1_macro'].mean():.4f} "
    f"± {results['train_f1_macro'].std():.4f}"
)

print(
    f"Accuracy: "
    f"{results['train_accuracy'].mean():.4f} "
    f"± {results['train_accuracy'].std():.4f}"
)

Resultados por fold:

Fold 1: F1 Macro = 0.4582 | Accuracy = 0.4596
Fold 2: F1 Macro = 0.4669 | Accuracy = 0.4680
Fold 3: F1 Macro = 0.4650 | Accuracy = 0.4652
Fold 4: F1 Macro = 0.4574 | Accuracy = 0.4577
Fold 5: F1 Macro = 0.4666 | Accuracy = 0.4669

Resultado da validação cruzada:
Macro F1: 0.4628 ± 0.0042
Accuracy: 0.4635 ± 0.0041

Treino:
Macro F1: 0.8575 ± 0.0014
Accuracy: 0.8576 ± 0.0014


## 6. Rotulação do conjunto de teste

In [ ]:
# ============================================================
# 6. Classificação do conjunto de teste externo
# ============================================================

test_df = pd.read_excel("test1.xlsx")

# Textos que serão classificados
X_new = test_df["resp_text"]

# Aplicar o mesmo TF-IDF treinado anteriormente
X_new_tfidf = vect.transform(X_new)

# Fazer as previsões
predicted_test = clf.predict(X_new_tfidf)

# Adicionar as previsões à coluna clarity
test_df["clarity"] = predicted_test

# Conferir resultado
print(test_df["clarity"].value_counts())
print(test_df.head())

clarity
c234    313
c5      306
c1      281
Name: count, dtype: int64
                                           resp_text clarity
0   Prezada Hellen,    Dentre as pesquisas domici...    c234
1   Senhor Francisco,  O Serviço de Informações a...    c234
2   Prezada Srª. Raylane, Em resposta a sua solic...      c5
3   Prezada Senhora,  De acordo com o art.13 do D...    c234
4   Prezado(a) Senhor(a),   Em atendimento ao ped...    c234


In [ ]:
# Salva o arquivo rotulado
test_df.to_excel("test1_rotulado.xlsx", index=False)